In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA 
from sklearn.metrics import silhouette_score

In [ ]:
df = pd.read_csv("university_admission.csv")

In [ ]:
df.head()

In [ ]:
df.shape

In [ ]:
df.info()

In [ ]:
df.isnull().sum()

In [ ]:
# df.duplicated().sum()

In [ ]:
df = df.drop(columns=["Chance_of_Admission"])

In [ ]:
df

In [ ]:
df.hist(figsize=(12,8))
plt.show()

In [ ]:
plt.figure(figsize=(10,8))
sns.heatmap(df.corr(),annot=True)
plt.show()

In [ ]:
X = df

In [ ]:
# Feature scaling 
scaler = StandardScaler()
X = scaler.fit_transform(df)

In [ ]:
#Elbow Method

In [ ]:
wcss = []

for k in range(1, 11):

    model = KMeans(
        n_clusters=k,
        random_state=42,
        n_init=10
    )

    model.fit(X)

    wcss.append(model.inertia_)

In [ ]:
plt.figure(figsize=(8,5))
plt.plot(range(1,11), wcss, marker="o")
plt.xlabel("Clusterlar soni: ")
plt.ylabel("WCSS")
plt.title("Elbow method")
plt.grid(True)
plt.show()

In [ ]:
# model training

kmeans = KMeans(n_clusters=4, random_state=42, n_init=10)
clusters = kmeans.fit_predict(X)

In [ ]:
df["Clusters"] = clusters

In [ ]:
df

In [ ]:
df["Clusters"].value_counts()

In [ ]:
df.groupby("Clusters").mean(numeric_only=True)

In [ ]:
score=silhouette_score(X, clusters)
print(score)

In [ ]:
# # cluster markazlari

# centers = scaler.inverse_transform(kmeans.cluster_centers_)

# centers = pd.DataFrame(centers, 
#                        columns=df.drop(columns="Clusters").columns)

# centers

In [ ]:
df

In [ ]:
pca = PCA(n_components=2, random_state=42) # principal component analysis
X_pca = pca.fit_transform(X)

print(f"Bu 2 komponent ma'lumotdagi umumiy o'zgaruvchanlikning "
      f"{pca.explained_variance_ratio_.sum()*100:.1f}%ini tushuntiradi")

In [ ]:
plt.figure(figsize=(9,7))
sochilish = plt.scatter(X_pca[:, 0], X_pca[:, 1], c=df["Clusters"], cmap="viridis", alpha=0.7)
plt.xlabel("1-komponenta")
plt.ylabel("2-komponenta")
plt.title("Klasterlarning PCA orqali 2D vizualizatsiyasi")
plt.colorbar(sochilish, label="Klaster")
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
# Klaster markazlarini asl (scale qilinmagan) birliklarga qaytaramiz
markazlar = scaler.inverse_transform(kmeans.cluster_centers_)

markazlar_df = pd.DataFrame(
    markazlar,
    columns=df.drop(columns="Clusters").columns
)
markazlar_df.index.name = "Klaster"
markazlar_df.round(2)

In [ ]:
# Uchta gipotetik yangi abituriyent (hali ma'lumotlar bazasida yo'q)
yangi_abituriyentlar = pd.DataFrame({
    "GRE_Score":          [335, 300, 318],
    "TOEFL_Score":        [116, 96,  108],
    "University_Rating":  [5,   2,   3],
    "SOP":                [4.5, 2.0, 3.5],
    "LOR":                [4.5, 2.5, 3.0],
    "CGPA":               [9.6, 7.2, 8.5],
    "Research":           [1,   0,   1],
})

yangi_abituriyentlar


In [ ]:
yangi_X = scaler.transform(yangi_abituriyentlar)

bashorat = kmeans.predict(yangi_X)

yangi_abituriyentlar["Bashorat_qilingan_klaster"] = bashorat
yangi_abituriyentlar

In [ ]:
markazlar_df